# Install Requirements

In [5]:
!pip install requests
!conda install pandas

mambajs 0.21.4

Process pip requirements ...

Requirement requests already satisfied.
mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 0.831 seconds
All requested packages already installed.


# Configuration

In [6]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Account
EQUITY = 10_000
RISK_PER_TRADE = 0.10
LEVERAGE = 1.0

# Costs/Fees
TAKER_FEE_RATE = 0.0010
SLIPPAGE_RATE = 0.0005

# Data
BASE_URL = "https://data-api.binance.vision"
SYMBOL = "BTCUSDT"
BAR_SIZE = "1h"

# Strategy Parameters
HORIZON = 37
THRESHOLD_MULTIPLIER = 1.0          # multiply round-trip cost
ALLOW_SHORTS = True

# Model Parameters
LR = 0.00325
MODEL_L2 = 0.425

# For fixed-length Binance intervals used here.
# Add other intervals if you use them.
BAR_SECONDS = {
    "1m": 60,
    "3m": 3 * 60,
    "5m": 5 * 60,
    "15m": 15 * 60,
    "30m": 30 * 60,
    "1h": 60 * 60,
    "2h": 2 * 60 * 60,
    "4h": 4 * 60 * 60,
    "6h": 6 * 60 * 60,
    "8h": 8 * 60 * 60,
    "12h": 12 * 60 * 60,
    "1d": 24 * 60 * 60,
}

if BAR_SIZE not in BAR_SECONDS:
    raise ValueError(
        f"Unsupported BAR_SIZE={BAR_SIZE!r}. "
        f"Supported values: {list(BAR_SECONDS)}"
    )

BAR_SECONDS_VALUE = BAR_SECONDS[BAR_SIZE]

# Wait a few seconds after the expected close before requesting data.
CLOSE_GRACE_SECONDS = 3

# Keep the last handled candle timestamp across kernel restarts
# if this file remains in the notebook's working directory.
LAST_PROCESSED_FILE = Path("last_processed_bar.txt")

# Existing position state. This survives cell reruns, but not kernel restarts.
if "_wf_previous_position" not in globals():
    _wf_previous_position = 0
    _wf_previous_size_usd = 0.0

# Paper/simulation convenience only. In live trading, reconcile position state
# from confirmed fills rather than assuming a generated signal was filled.
ASSUME_SIGNAL_FILLED = True

# Load Closed Data

In [7]:
def load_closed_data(symbol, base_url, interval, limit=1000, max_attempts=3):
    """
    Load recent Binance klines and return only candles that are closed.

    Retries are bounded. HTTP 429/418 are raised immediately so the caller
    can back off instead of repeatedly hitting the API.
    """
    if not 1 <= limit <= 1000:
        raise ValueError("'limit' must be between 1 and 1000.")

    url = f"{base_url.rstrip('/')}/api/v3/klines"
    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": limit,
    }

    last_error = None

    for attempt in range(max_attempts):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=(3, 10),  # connect timeout, response timeout
            )

            # Don't automatically retry rate limits here. The outer loop
            # will respect Retry-After and wait before trying again.
            if response.status_code in (418, 429):
                response.raise_for_status()

            if response.status_code in (500, 502, 503, 504):
                response.raise_for_status()

            response.raise_for_status()
            rows = response.json()

            if not rows:
                raise ValueError("No data returned from Binance.")

            columns = [
                "open_time",
                "open",
                "high",
                "low",
                "close",
                "volume",
                "close_time",
                "quote_asset_volume",
                "number_of_trades",
                "taker_buy_base_asset_volume",
                "taker_buy_quote_asset_volume",
                "ignore",
            ]

            data = pd.DataFrame(rows, columns=columns)
            data["open_time"] = pd.to_datetime(
                data["open_time"], unit="ms", utc=True
            )
            data["close_time"] = pd.to_datetime(
                data["close_time"], unit="ms", utc=True
            )

            numeric_columns = [
                "open",
                "high",
                "low",
                "close",
                "volume",
                "quote_asset_volume",
                "number_of_trades",
                "taker_buy_base_asset_volume",
                "taker_buy_quote_asset_volume",
            ]

            for column in numeric_columns:
                data[column] = pd.to_numeric(data[column], errors="coerce")

            # Leave a small grace period for exchange/data-feed timing.
            cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(seconds=2)
            data = data.loc[data["close_time"] < cutoff].copy()

            if data.empty:
                raise RuntimeError(
                    "Binance returned no fully closed candles yet."
                )

            data = (
                data.sort_values("open_time")
                    .drop_duplicates("open_time")
                    .set_index("open_time")
            )

            return data

        except requests.HTTPError:
            # Don't retry 429/418 here; caller handles Retry-After.
            raise

        except (requests.RequestException, ValueError, RuntimeError) as exc:
            last_error = exc

            if attempt + 1 >= max_attempts:
                break

            # Bounded retry delay for temporary network/server/data issues.
            time.sleep(1 + attempt)

    raise RuntimeError(
        f"Could not load closed candles after {max_attempts} attempts: "
        f"{last_error}"
    )

# Build Features

In [8]:
def build_features(raw_df, horizon=30):
    df = raw_df.copy()

    # Past multi-horizon log returns
    return_windows = [125, 150, 175, 200, 225, 250]
    for window in return_windows:
        df[f"ret_{window}"] = np.log(df["close"] / df["close"].shift(window))

    # Target - Future log return for "horizon"
    df["target"] = np.log(
        df["close"].shift(-horizon) / df["close"]
    )

    feature_cols = (
        [f"ret_{w}" for w in return_windows]
    )

    df = df.copy()

    return df, feature_cols, "target"

# Model

In [9]:
class LinearRegressionModel:
    def __init__(self, n_features, lr=0.001, model_l2=0.0):
        self.w = np.zeros(n_features, dtype=float)
        self.b = 0.0
        self.lr = lr
        self.model_l2 = model_l2

    def predict(self, x):
        x = np.asarray(x, dtype=float).reshape(-1)
        return float(np.dot(self.w, x) + self.b)

    def update(self, x, y):
        x = np.asarray(x, dtype=float).reshape(-1)
        y = float(y)

        pred = self.predict(x)
        err = pred - y

        self.w -= self.lr * (err * x + self.model_l2 * self.w)
        self.b -= self.lr * err

        return err

# Prediction To Position

In [10]:
def prediction_to_position(pred, threshold_multiplier, taker_fee_rate, slippage_rate, allow_shorts=True):
    one_way_cost = taker_fee_rate + slippage_rate
    threshold = abs(threshold_multiplier * one_way_cost)
    if pred > threshold:
        return 1
    elif allow_shorts and pred < -threshold:
        return -1
    else:
        return 0

# Make Signal From Closed Data

In [11]:
def make_signal_from_closed_data(raw_closed_df):
    global _wf_previous_position, _wf_previous_size_usd

    df, feature_cols, target_col = build_features(raw_closed_df, HORIZON)

    if len(df) <= HORIZON:
        raise ValueError(
            f"Need more than {HORIZON} rows; got {len(df)}."
        )

    X = df[feature_cols].to_numpy(dtype=float)
    y = df[target_col].to_numpy(dtype=float)

    model = LinearRegressionModel(
        n_features=len(feature_cols),
        lr=LR,
        model_l2=MODEL_L2,
    )

    # Replay history causally: at row i, the label from i - HORIZON
    # is now known.
    for i in range(len(df)):
        matured_idx = i - HORIZON

        if matured_idx >= 0:
            x_matured = X[matured_idx]
            y_matured = y[matured_idx]

            if np.isfinite(y_matured) and np.all(np.isfinite(x_matured)):
                model.update(x_matured, float(y_matured))

    x_current = X[-1]

    if not np.all(np.isfinite(x_current)):
        raise ValueError("Latest row has missing/non-finite features.")

    pred = float(np.asarray(model.predict(x_current)).reshape(-1)[0])
    pred = float(np.clip(pred, -0.2, 0.2))

    desired_position = int(
        prediction_to_position(
            pred,
            THRESHOLD_MULTIPLIER,
            TAKER_FEE_RATE,
            SLIPPAGE_RATE,
            ALLOW_SHORTS,
        )
    )

    timestamp = df.index[-1]
    price = float(df["close"].iloc[-1])

    if not np.isfinite(price) or price <= 0:
        raise ValueError(f"Invalid latest close price: {price}")

    previous_position = int(_wf_previous_position)
    previous_size_usd = float(_wf_previous_size_usd)

    # This is target notional, not a stop-loss-based risk amount.
    target_size_usd = EQUITY * LEVERAGE * RISK_PER_TRADE

    if desired_position == previous_position:
        action = "HOLD" if desired_position != 0 else "STAY_FLAT"
        signal_size_usd = previous_size_usd
        order_signed_quantity = 0.0

    elif desired_position == 0:
        action = "CLOSE"
        signal_size_usd = 0.0
        order_signed_quantity = (
            -previous_position * previous_size_usd / price
        )

    else:
        if previous_position == 0:
            action = "OPEN_LONG" if desired_position == 1 else "OPEN_SHORT"
        else:
            action = (
                "REVERSE_TO_LONG"
                if desired_position == 1
                else "REVERSE_TO_SHORT"
            )

        signal_size_usd = target_size_usd

        current_signed_quantity = (
            previous_position * previous_size_usd / price
        )
        target_signed_quantity = desired_position * target_size_usd / price
        order_signed_quantity = (
            target_signed_quantity - current_signed_quantity
        )

    signal = {
        "timestamp": timestamp,
        "prediction": pred,
        "previous_position": previous_position,
        "position": desired_position,
        "action": action,
        "size_usd": float(signal_size_usd),
        "order_signed_quantity": float(order_signed_quantity),
    }

    # For paper trading only: assume the signal filled.
    # For live trading, update this state after confirmed fills instead.
    if ASSUME_SIGNAL_FILLED and desired_position != previous_position:
        _wf_previous_position = desired_position
        _wf_previous_size_usd = (
            target_size_usd if desired_position != 0 else 0.0
        )

    return signal

# Read Last Processed Timestamp

In [ ]:
def read_last_processed_timestamp():
    if not LAST_PROCESSED_FILE.exists():
        return None

    text = LAST_PROCESSED_FILE.read_text().strip()
    if not text:
        return None

    return pd.Timestamp(text)


def save_last_processed_timestamp(timestamp):
    LAST_PROCESSED_FILE.write_text(pd.Timestamp(timestamp).isoformat())


def sleep_until_next_bar():
    now = time.time()

    next_boundary = (
        (int(now) // BAR_SECONDS_VALUE + 1) * BAR_SECONDS_VALUE
        + CLOSE_GRACE_SECONDS
    )

    sleep_seconds = max(1.0, next_boundary - now)
    print(f"Waiting about {sleep_seconds:.0f}s for the next bar check.")
    time.sleep(sleep_seconds)


def retry_after_seconds(exc):
    response = getattr(exc, "response", None)
    if response is None:
        return None

    value = response.headers.get("Retry-After")
    if value is None:
        return None

    try:
        return max(1.0, float(value))
    except ValueError:
        return None


last_processed_bar = read_last_processed_timestamp()
error_backoff = 5

print("Starting bar monitor.")
print("Previously processed bar:", last_processed_bar)

while True:
    try:
        # One full-history request per check. No separate timestamp-check request.
        raw_closed_df = load_closed_data(
            SYMBOL,
            BASE_URL,
            BAR_SIZE,
            limit=1000,
        )

        latest_closed_timestamp = raw_closed_df.index[-1]

        if (
            last_processed_bar is None
            or latest_closed_timestamp > last_processed_bar
        ):
            signal = make_signal_from_closed_data(raw_closed_df)

            print("New closed bar:", latest_closed_timestamp)
            display(signal)

            # Save only after the signal function succeeds.
            save_last_processed_timestamp(latest_closed_timestamp)
            last_processed_bar = latest_closed_timestamp
            error_backoff = 5
        else:
            print("No new closed bar.")

        sleep_until_next_bar()

    except KeyboardInterrupt:
        print("Stopped by user.")
        break

    except Exception as exc:
        wait_seconds = retry_after_seconds(exc)

        if wait_seconds is None:
            wait_seconds = error_backoff
            error_backoff = min(error_backoff * 2, 300)

        print(
            f"Data check failed: {exc}. "
            f"Waiting {wait_seconds:.0f}s before retrying."
        )
        time.sleep(wait_seconds)

Starting bar monitor.
Previously processed bar: 2026-10-07 13:00:00+00:00
No new closed bar.
Waiting about 1960s for the next bar check.
